In [1]:
import sys
import cv2
import time
from matplotlib import pyplot as plt
from tqdm import tqdm, trange
import numpy as np
import pandas as pd
import pickle
import random
import copy
import json

import os
import shutil
from PIL import Image, ImageDraw

import torch
from torch.utils.data import Dataset
from torchvision.transforms.functional import to_tensor, normalize
from torchvision import transforms

from coco.dataset import COCODataset, COCODatasetRandom
from utils import *
sys.path.append("..")

In [2]:
test_annotations = '../datasets/COCO18_dset_for_CRT_training/coco18_test_deepgaze1.json'
test_imagedir = '../datasets/COCO18_dset_for_CRT_training/coco18_test_deepgaze_resize/'

In [3]:
import json
import pickle

with open(test_annotations, 'rb') as file:
    train_metadata = json.load(file)
    
categories_id_to_name = {}

# _____ CREATE A LOOKUP TABLE FOR CATEGORY ID TO CATEGORY NAME _____
for info in train_metadata['categories']:
    categories_id_to_name[info['id']] = info['name']
# _____ CREATE A LOOKUP TABLE FOR CATEGORY ID TO CATEGORY NAME _____
    
    
    
    
    
# _____ CREATE A LOOKUP TABLE FOR INDEXES IN THE SAME CATEGORIES _____
category_idx_dict = {}

for i in range(len(train_metadata['annotations'])):
    
    info = train_metadata['annotations'][i]
    
    name = categories_id_to_name[info['category_id']]
    
    if name not in category_idx_dict:
        category_idx_dict[name] = []
        
    category_idx_dict[name].append(i)
# _____ CREATE A LOOKUP TABLE FOR INDEXES IN THE SAME CATEGORIES _____



with open('../datasets/COCO18_dset_for_CRT_training/coco18_idx_test612.pkl', 'wb') as file:
    pickle.dump(category_idx_dict, file, protocol=pickle.HIGHEST_PROTOCOL)

In [4]:




category_dic_dir = '../datasets/COCO18_dset_for_CRT_training/coco18_idx_test612.pkl'
image_list = os.listdir(test_imagedir)
input_images = COCODatasetRandom(test_annotations, test_imagedir, image_size =(224,224), category_dic_dir = category_dic_dir, normalize_means=[0.485, 0.456, 0.406], normalize_stds=[0.229, 0.224, 0.225])

-------------------------------
Annotation Counts
-------------------------------
chair                        43
fork                         41
sink                         51
tv                           50
bowl                         26
car                          20
clock                        23
cup                          49
keyboard                     33
knife                        24
laptop                       23
mouse                        19
oven                         19
potted plant                 28
toilet                       29
bottle                       30
stop sign                    25
microwave                    27
Total                       560
-------------------------------



In [5]:
# define IVSN model
class IVSN(nn.Module):
  def __init__(self, model):
      super(IVSN, self).__init__()
      self.features = nn.Sequential(*list(model.children())[0][:30])
      for param in self.features.parameters():
        param.requires_grad_ = False

  def forward(self, x):
      x = self.features(x)
      return x

from torch.nn.modules.conv import Conv2d
ConvSize, NumTemplates, Mylayer = 1, 512, 31
TotalTrials, targetsize, stimulisize = 600, 32, 224
MMconv = Conv2d(NumTemplates, 1, kernel_size = (ConvSize, ConvSize), stride = (1, 1), padding = (1, 1))

In [6]:
model_vgg = models.vgg16(pretrained=True)
model_ivsn = IVSN(model_vgg)

/Users/nguyenduysmacbook/Desktop/AnhHaiFYP/FYP_venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nguyenduysmacbook/Desktop/AnhHaiFYP/FYP_venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
num_pics, size, image_size, shorter_side = len(input_images), 48, (320, 512), 128
IVSN_attention_map, scanpath = {}, {}
IVSN_res = list()

with torch.no_grad():
    for id in trange(num_pics):
        context_images, target_images, bbox, labels_cpu = input_images[id]
        # compute the ratio between width and height of the target, and set shorter side 128
        # aspect_ration = (bbox[2].item()*image_size[1])/(bbox[3].item()*image_size[0])
        tg_resize = (shorter_side, shorter_side)

        # get attention map from IVSN model
        context_images = transforms.Resize(image_size)(context_images)
        context_ivsn = context_images.view(1, 3, 320, 512)
        target_feedIVSN = transforms.Resize(tg_resize)(target_images)
        target_ivsn = target_feedIVSN.view(1, 3, tg_resize[0], tg_resize[1])
        cont_output_ivsn = model_ivsn(context_ivsn)
        tg_output_ivsn = model_ivsn(target_ivsn)
        MMconv.weight = torch.nn.Parameter(tg_output_ivsn)
        attention_IVSN = MMconv.forward(cont_output_ivsn)
        attention_IVSN = attention_IVSN.detach().squeeze(0)

        # # generate key for vit attention map
        label_name = input_images.idx2label[labels_cpu]

        tg_loc = bbox_cordinates(bbox, 512, 320)

        # process IVSN attention map
        mask_IVSN = transforms.Resize(image_size)(attention_IVSN)
        mask_IVSN = torch.divide(mask_IVSN, mask_IVSN.max())

        IVSN_attention_map[id] = copy.deepcopy(mask_IVSN)

        IVSN_num, path = searchProcesswithPath(tg_loc, mask_IVSN, image_size, size)
        scanpath[id] = path
        IVSN_res.append(IVSN_num)
        print('IVSN_' + label_name +  "_" + str(id) + ': ' + str(IVSN_num), end = '\t')


  0%|▍                                                                                                                                         | 2/560 [00:00<01:16,  7.25it/s]

IVSN_chair_0: 16	IVSN_chair_1: 2	

  1%|▉                                                                                                                                         | 4/560 [00:00<01:12,  7.67it/s]

IVSN_chair_2: 26	IVSN_chair_3: 4	

  1%|█▍                                                                                                                                        | 6/560 [00:00<01:20,  6.91it/s]

...

IVSN_microwave_554: 9	IVSN_microwave_555: 80	

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 558/560 [01:10<00:00,  8.06it/s]

IVSN_microwave_556: 2	IVSN_microwave_557: 2	

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 560/560 [01:10<00:00,  7.89it/s]

IVSN_microwave_558: 3	IVSN_microwave_559: 2	

In [8]:
IVSN_res_fig5 = {}
IVSN_res_fig5['search_list'] = IVSN_res
IVSN_res_fig5['attention_map'] = IVSN_attention_map
IVSN_res_fig5['scanpath'] = scanpath

In [9]:
with open("../results/COCO/[Fig5]coco_IVSN_res.pkl", "wb") as tf:
    pickle.dump(IVSN_res_fig5, tf)

In [10]:
IVSN_accu_OR = model_performance(IVSN_res, len(IVSN_res))
IVSN_accu_OR[:11]

[0,
 0.0,
 np.float64(0.4089285714285714),
 np.float64(0.4785714285714286),
 np.float64(0.5267857142857143),
 np.float64(0.5589285714285714),
 np.float64(0.6071428571428571),
 np.float64(0.625),
 np.float64(0.6446428571428572),
 np.float64(0.6714285714285714),
 np.float64(0.6839285714285714)]

In [11]:
with open("../results/COCO/coco_IVSN_OR_accu_performance.pkl", "wb") as tf:
    pickle.dump(IVSN_accu_OR, tf)